In [1]:
import pandas as pd

In [2]:
df = pd.read_json("nas_results.json")
df

,stem_op_idx,stem_ch,b2_op_idx,b2_ch,b2_n,b2_n_aware,b2_skip,b2_inc_split_idx,b3_op_idx,b3_ch,...,b5_final_op_idx,neck_ch,activation,proxy_map50,est_ram,est_flash,generation,eval_id,parameters,age
0,0,16,0,16,1,3,True,1,0,32,...,1.0,24,ReLU,0.122716,466944,300520,0,g00_c001,77957,NaN
1,4,16,1,24,2,6,False,1,2,24,...,1.0,16,Hardswish,0.163850,466944,539552,0,g00_c002,128749,NaN
2,1,12,0,24,1,6,False,2,0,32,...,1.0,16,Hardswish,0.180192,368640,426240,0,g00_c003,130742,NaN
3,1,12,0,12,1,6,True,3,3,40,...,1.0,24,ReLU,0.166106,368640,475296,0,g00_c004,114366,NaN
4,4,8,1,12,1,6,True,1,0,32,...,1.0,24,Hardswish,0.158689,270336,319976,0,g00_c005,93549,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,0,16,1,16,3,6,False,3,4,24,...,NaN,24,SiLU,0.225905,466944,579468,4,g04_c072,190549,0.0
72,1,16,1,24,1,9,False,1,5,24,...,NaN,32,SiLU,0.105969,466944,573196,4,g04_c073,230368,0.0
73,0,16,1,16,1,6,True,3,4,24,...,NaN,24,SiLU,0.175618,466944,575572,5,g05_c074,189621,0.0
74,0,16,4,12,1,6,False,3,3,24,...,NaN,24,SiLU,0.193269,466944,514308,5,g05_c075,212501,0.0


In [11]:
from nas import Architecture, pareto_score, arch_to_yaml, estimate_hardware_cost

In [4]:
import json
all_results = json.load(open("nas_results.json"))
archs = [Architecture.from_dict(d) for d in all_results]
df["pareto_score"] = [pareto_score(arch, 500_000, 600_000) for arch in archs]
# df = df.sort_values("pareto_score", ascending=False)
df.head()

,stem_op_idx,stem_ch,b2_op_idx,b2_ch,b2_n,b2_n_aware,b2_skip,b2_inc_split_idx,b3_op_idx,b3_ch,...,neck_ch,activation,proxy_map50,est_ram,est_flash,generation,eval_id,parameters,age,pareto_score
0,0,16,0,16,1,3,True,1,0,32,...,24,ReLU,0.122716,466944,300520,0,g00_c001,77957,NaN,0.082189
1,4,16,1,24,2,6,False,1,2,24,...,16,Hardswish,0.163850,466944,539552,0,g00_c002,128749,NaN,0.103210
2,1,12,0,24,1,6,False,2,0,32,...,16,Hardswish,0.180192,368640,426240,0,g00_c003,130742,NaN,0.127536
3,1,12,0,12,1,6,True,3,3,40,...,24,ReLU,0.166106,368640,475296,0,g00_c004,114366,NaN,0.116208
4,4,8,1,12,1,6,True,1,0,32,...,24,Hardswish,0.158689,270336,319976,0,g00_c005,93549,NaN,0.124486


In [5]:
sorted_df = df.sort_values("pareto_score", ascending=False)
sorted_df.head(10)

,stem_op_idx,stem_ch,b2_op_idx,b2_ch,b2_n,b2_n_aware,b2_skip,b2_inc_split_idx,b3_op_idx,b3_ch,...,neck_ch,activation,proxy_map50,est_ram,est_flash,generation,eval_id,parameters,age,pareto_score
60,0,8,2,12,1,3,False,3,4,32,...,16,SiLU,0.194807,307200,449300,4,g04_c061,123581,NaN,0.144312
43,0,16,2,12,1,6,False,3,4,40,...,24,Hardswish,0.210229,466944,285352,2,g02_c044,59209,NaN,0.141332
39,0,16,2,12,1,6,False,3,4,40,...,24,Hardswish,0.210229,466944,285352,2,g02_c040,59209,NaN,0.141332
33,0,16,2,12,1,6,False,3,4,40,...,24,Hardswish,0.210229,466944,285352,2,g02_c034,59209,NaN,0.141332
49,0,16,2,12,1,6,False,3,4,40,...,24,Hardswish,0.210229,466944,285352,3,g03_c050,59209,NaN,0.141332
44,0,16,2,12,1,6,False,3,4,40,...,24,Hardswish,0.210229,466944,285352,2,g02_c045,59209,NaN,0.141332
71,0,16,1,16,3,6,False,3,4,24,...,24,SiLU,0.225905,466944,579468,4,g04_c072,190549,0.0,0.140796
67,0,16,1,16,3,6,False,3,4,24,...,24,SiLU,0.225905,466944,579468,4,g04_c068,190549,0.0,0.140796
70,0,16,4,12,3,3,False,3,0,40,...,24,Hardswish,0.221820,466944,573596,4,g04_c071,206397,0.0,0.138467
66,0,16,4,12,3,3,False,3,0,40,...,24,Hardswish,0.221820,466944,573596,4,g04_c067,206397,0.0,0.138467


In [6]:
best_arch = sorted_df.iloc[0]
best_arch = [a for a in archs if a.eval_id == best_arch["eval_id"]][0]
b2_op_idx = best_arch.b2_op_idx
b3_op_idx = best_arch.b3_op_idx
b4_op_idx = best_arch.b4_op_idx

In [7]:
with open("/tmp/best_arch.yaml", "w") as f:
    yaml = arch_to_yaml(best_arch)
    f.write(yaml)
    print(yaml)

# NAS arch  id=g04_c061
nc: 1

backbone:
  - [-1, 1, Conv, [8, 3, 1]]  # stem Conv 3->8
  - [-1, 1, nn.MaxPool2d, [2, 2, 0]]  # 128->64
  - [-1, 3, C2f, [12, True]]  # B2 C2f n=3
  - [-1, 1, nn.MaxPool2d, [2, 2, 0]]  # 64->32
  - [-1, 1, GhostConv, [32, 3, 1]]  # B3-1 GhostConv->32ch
  - [-1, 1, GhostConv, [32, 3, 1]]  # B3-2 GhostConv->32ch
  - [-1, 1, nn.MaxPool2d, [2, 2, 0]]  # 32->16
  - [-1, 1, DSConv, [80, 3, 1]]  # B4-1 DSConv->80ch
  - [-1, 1, DSConv, [80, 3, 1]]  # B4-2 DSConv->80ch
  - [6, 1, Conv, [80, 1, 1]]  # skip proj 32->80
  - [[8, 9], 1, Add, []]  # skip to B4
  - [-1, 1, nn.MaxPool2d, [2, 2, 0]]  # 16->8
  - [-1, 1, GhostConv, [96, 3, 1]]  # B5-1 GhostConv->96ch
  - [-1, 1, GhostConv, [96, 3, 1]]  # B5-2 GhostConv->96ch
  - [11, 1, Conv, [96, 1, 1]]  # skip proj 80->96
  - [[13, 14], 1, Add, []]  # skip to B5
  - [-1, 1, nn.MaxPool2d, [2, 2, 0]]  # 8->4

head:
  - [-1, 1, Conv, [16, 1, 1]]  # neck 96->16ch
  - [[-1], 1, Detect, [nc]]  # detect 4×4


In [15]:
from ultralytics import YOLO
model = YOLO("temp.yaml")
model.info()
model.export(
        format="tflite",
        imgsz=128,
        int8=True,
        data="person.yaml",  # calibration data
        nms=True,
        simplify=True,
        dynamic=False
    )

the length of ch is: {1}
temp summary: 112 layers, 198669 parameters, 198665 gradients
Ultralytics YOLOv8.1.29 🚀 Python-3.10.11 torch-2.10.0+cu128 CPU (AMD Ryzen 7 5800H with Radeon Graphics)
temp summary (fused): 84 layers, 197557 parameters, 0 gradients

PyTorch: starting from 'temp.yaml' with input shape (1, 3, 128, 128) BCHW and output shape(s) (1, 5, 64) (0.0 MB)

TensorFlow SavedModel: starting export with tensorflow 2.21.0...
WARNING ⚠️ tensorflow<=2.13.1 is required, but tensorflow==2.21.0 is currently installed https://github.com/ultralytics/ultralytics/issues/5161

ONNX: starting export with onnx 1.21.0 opset 10...
ONNX: simplifying with onnxsim v0.6.2...
ONNX: export success ✅ 0.4s, saved as 'temp.onnx' (0.8 MB)
TensorFlow SavedModel: collecting INT8 calibration images from 'data=person.yaml'


Scanning D:\VideoSummarizer\datasets\coco-2017\val\labels.cache... 2693 images, 480 backgrounds, 0 corrupt: 100%|██████████| 2693/2693 [00:00<?, ?it/s]


TensorFlow SavedModel: starting TFLite export with onnx2tf 1.17.5...

Automatic generation of each OP name started ========================================
Automatic generation of each OP name complete!

Model loaded ========================================================================

Model conversion started ============================================================
ERROR: The trace log is below.
Traceback (most recent call last):
  File "d:\VideoSummarizer\venv\lib\site-packages\onnx2tf\utils\common_functions.py", line 288, in print_wrapper_func
    result = func(*args, **kwargs)
  File "d:\VideoSummarizer\venv\lib\site-packages\onnx2tf\utils\common_functions.py", line 361, in inverted_operation_enable_disable_wrapper_func
    result = func(*args, **kwargs)
  File "d:\VideoSummarizer\venv\lib\site-packages\onnx2tf\ops\Conv.py", line 453, in make_node
    conv_bias(
  File "d:\VideoSummarizer\venv\lib\site-packages\onnx2tf\ops\Conv.py", line 302, in conv_bias
    tf.nn.convolut

SystemExit: 1

In [14]:
modified_arch = {'stem_op_idx': 0,
 'stem_ch': 8,
 'b2_op_idx': 4,
 'b2_ch': 12,
 'b2_n': 2,
 'b2_n_aware': 3,
 'b2_skip': False,
 'b2_inc_split_idx': 3,
 'b3_op_idx': 4,
 'b3_ch': 32,
 'b3_n': 2,
 'b3_n_aware': 3,
 'b3_skip': False,
 'b3_inc_split_idx': 1,
 'b4_op_idx': 1,
 'b4_ch': 80,
 'b4_n': 2,
 'b4_n_aware': 3,
 'b4_skip': True,
 'b4_inc_split_idx': 0,
 'b5_op_idx': 2,
 'b5_ch': 96,
 'b5_n': 2,
 'b5_n_aware': 3,
 'b5_skip': True,
 'b5_inc_split_idx': 1,
 'b5_final_op_idx': None,
 'neck_ch': 16,
 'activation': 'SiLU',
 'proxy_map50': 0.242,
 'est_ram': 270336,
 'est_flash': 797416,
 'generation': 4,
 'eval_id': 'g04_c061',
 'parameters': 123581,
 'age': 0}

ar = Architecture.from_dict(modified_arch)
estimate_hardware_cost(ar)

pareto_score(ar, 500_000, 600_000)


0.17058476746666665

In [91]:
import os
os.environ["WANDB_MODE"] = "disabled"

model = YOLO("temp.yaml")
results = model.train(
            data     = "person.yaml",
            imgsz    = 128,
            epochs   = 2,
            batch    = 64,
            workers  = 3,
            optimizer= "Adam",
            lr0      = 1e-3,
            cos_lr   = True,
            project  = "best_arch",
            name     = f"nas",
            exist_ok = True,
            verbose  = False,
            save     = True,
            device   = "0",
        )

the length of ch is: {1}
New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.1.29 🚀 Python-3.10.11 torch-2.10.0+cu128 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: -1
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.
